In [1]:
import pandas as pd
import os
import random
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from xgboost import XGBClassifier
from sklearn.feature_selection import VarianceThreshold
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix, classification_report
import seaborn as sns
import numpy as np
from scipy.stats import entropy
import warnings
from sklearn.feature_selection import SelectKBest, f_classif

In [2]:
prefixes = [
    # 'Left_Hallux_raw', 'Right_Hallux_raw',
    # 'Left_Toes_raw', 'Right_Toes_raw',
    # 'Left_Met1_raw', 'Left_Met3_raw', 'Left_Met5_raw',
    # 'Right_Met1_raw', 'Right_Met3_raw', 'Right_Met5_raw', 
    # 'Left_Arch_raw', 'Right_Arch_raw',
    # 'Left_Heel_R_raw', 'Left_Heel_L_raw', 
    # 'Right_Heel_L_raw', 'Right_Heel_R_raw',

    "acceleration_Pelvis_x","acceleration_Pelvis_y","acceleration_Pelvis_z",
    # "acceleration_RightForeArm_x","acceleration_RightForeArm_y","acceleration_RightForeArm_z",
    "acceleration_RightUpperLeg_x","acceleration_RightUpperLeg_y","acceleration_RightUpperLeg_z",
    "acceleration_RightLowerLeg_x","acceleration_RightLowerLeg_y","acceleration_RightLowerLeg_z",
    "acceleration_RightFoot_x","acceleration_RightFoot_y","acceleration_RightFoot_z",
    "acceleration_RightToe_x","acceleration_RightToe_y","acceleration_RightToe_z",
    "acceleration_LeftUpperLeg_x","acceleration_LeftUpperLeg_y","acceleration_LeftUpperLeg_z",
    "acceleration_LeftLowerLeg_x","acceleration_LeftLowerLeg_y","acceleration_LeftLowerLeg_z",
    "acceleration_LeftFoot_x","acceleration_LeftFoot_y","acceleration_LeftFoot_z",
    "acceleration_LeftToe_x","acceleration_LeftToe_y","acceleration_LeftToe_z",

    "angularVelocity_Pelvis_x","angularVelocity_Pelvis_y","angularVelocity_Pelvis_z",
    # "angularVelocity_RightForeArm_x","angularVelocity_RightForeArm_y","angularVelocity_RightForeArm_z",
    "angularVelocity_RightUpperLeg_x","angularVelocity_RightUpperLeg_y","angularVelocity_RightUpperLeg_z",
    "angularVelocity_RightLowerLeg_x","angularVelocity_RightLowerLeg_y","angularVelocity_RightLowerLeg_z",
    "angularVelocity_RightFoot_x","angularVelocity_RightFoot_y","angularVelocity_RightFoot_z",
    "angularVelocity_RightToe_x","angularVelocity_RightToe_y","angularVelocity_RightToe_z",
    "angularVelocity_LeftUpperLeg_x","angularVelocity_LeftUpperLeg_y","angularVelocity_LeftUpperLeg_z",
    "angularVelocity_LeftLowerLeg_x","angularVelocity_LeftLowerLeg_y","angularVelocity_LeftLowerLeg_z",
    "angularVelocity_LeftFoot_x","angularVelocity_LeftFoot_y","angularVelocity_LeftFoot_z",
    "angularVelocity_LeftToe_x","angularVelocity_LeftToe_y","angularVelocity_LeftToe_z",

    # # calculated from raw data
    # "elevation", "orientation_x", "orientation_y", "orientation_z", "elevation_diff",
    
    'participant_id',  'walk_mode', 'stepcount'
    # 'participant_id',  'walk_mode'
]

In [ ]:
# data_file = os.path.join("data_set", "combined_stat_freq_features(sliding window).csv")
data_file = os.path.join("data_set", "combined_stat_freq_features(gait cycles).csv")
df = pd.read_csv(data_file)

# remove NaNs
initial_row_count = df.shape[0]
df = df.dropna()
# Get the number of rows after dropping NaN values
final_row_count = df.shape[0]
# Calculate how many rows were dropped
rows_dropped = initial_row_count - final_row_count
# print(f"Number of rows dropped: {rows_dropped}")

# Randomly drop data points until the dataframe size reaches a predefined length
# ONLY RUN THE NEXT LINE IF USING THE SLIDING WINDOW APPROACH
# df = df.sample(n=29988, random_state=42).reset_index(drop=True)

# ALWAYS RUN THE REST
# Find columns whose names start with any of the specified prefixes
cols_to_keep = [col for col in df.columns if any(col.startswith(prefix) for prefix in prefixes)]
df = df[cols_to_keep]

# df = df.drop('window_start', axis=1)
# df = df.drop('window_end', axis=1)
# df = df.drop(columns=["stepcount"])

# Replace class names with integers
unique_vals = df["walk_mode"].unique()

mapping_dict = {val: idx for idx, val in enumerate(unique_vals)}


df["walk_mode"] = df["walk_mode"].map(mapping_dict)


# # move the Name column to the postition before the last, keeping the Task column as the last one
# col = df.pop("Name")
# df.insert(len(df.columns) - 1, col.name, col)

In [4]:
def reduce_features(train_set, features, variance_threshold=0.01, max_corr=0.80):
    """this is the feature reduction method

    Parameters:
    train_set (dataframe): train set used to reduce the features
    features (list): initial feature names
    variance_threshold (float): variance threshold used to identify quasi-constant features
    max_corr (float): threshold to keep non-correlated features

    Returns:
    reduced_features (list): reduced feature names
    """

    print("reducing features...")

    X_train = train_set[features].copy()
    # Removing Constant features using variance threshold
    constant_filter = VarianceThreshold(threshold=0)
    constant_filter.fit(X_train)
    # Get the names of the constant features
    constant_columns = [
        column
        for column in X_train.columns
        if column not in X_train.columns[constant_filter.get_support()]
    ]
    # Drop constant features
    # for constant_column in constant_columns:
    #     print("dropping constant feature {}".format(constant_column))
    X_train = X_train.drop(labels=constant_columns, axis=1)  # Reassign X_train

    # Drop quasi-constant features
    qconstant_filter = VarianceThreshold(threshold=variance_threshold)
    qconstant_filter.fit(X_train)
    # Get the names of the qconstant features
    qconstant_columns = [
        column
        for column in X_train.columns
        if column not in X_train.columns[qconstant_filter.get_support()]
    ]
    # Drop qconstant features
    # for qconstant_column in qconstant_columns:
    #     print("dropping quasi-constant feature {}".format(qconstant_column))
    X_train = X_train.drop(labels=qconstant_columns, axis=1)  # Reassign X_train

    # Drop duplicate features
    transpose = X_train.T
    unique_features = transpose.drop_duplicates(keep="first").T.columns
    duplicate_features = [x for x in X_train.columns if x not in unique_features]
    # for duplicate_feature in duplicate_features:
    #     print("dropping duplicate feature {}".format(duplicate_feature))
    X_train = X_train.drop(labels=duplicate_features, axis=1)
    # Drop correlated features
    X_train = train_set[[*unique_features]].copy()
    correlated_features = set()
    correlation_matrix = X_train.corr()
    # Add the columns with a correlation value of max_corr to the correlated_features set
    for i in range(len(correlation_matrix.columns)):
        for j in range(i):
            if abs(correlation_matrix.iloc[i, j]) > max_corr:
                colname = correlation_matrix.columns[i]
                correlated_features.add(colname)
    # for correlated_feature in correlated_features:
    #     print(
    #         "dropping feature correlated greater than {} {}".format(
    #             max_corr, correlated_feature
    #         )
    #     )
    X_train = X_train.drop(labels=correlated_features, axis=1)
    reduced_features = X_train.columns
    return reduced_features

In [5]:
def filter_features_by_distribution(df, features, class_col, threshold=0.1):
    """
    Filters features based on their ability to differentiate between classes by comparing their distributions.

    Parameters:
        df (pd.DataFrame): The dataset containing the data.
        features (list): List of feature names to evaluate.
        class_col (str): The column name representing the class labels.
        threshold (float): Minimum KL divergence to retain a feature.

    Returns:
        list: Final list of retained features.
    """
    retained_features = []

    for feature in features:
        # Separate the data by class
        class1_data = df[df[class_col] == 0][feature]
        class2_data = df[df[class_col] == 1][feature]

        # Check for zero variance
        if len(np.unique(class1_data)) <= 1 or len(np.unique(class2_data)) <= 1:
            print(f"Feature {feature} skipped due to zero variance.")
            continue

        # Estimate probability density functions
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=RuntimeWarning)
            class1_pdf, class1_bins = np.histogram(class1_data, bins=50, density=True)
            class2_pdf, class2_bins = np.histogram(class2_data, bins=50, density=True)

        # Avoid zero probabilities
        class1_pdf += 1e-9
        class2_pdf += 1e-9

        # Compute KL divergence
        kl_div = entropy(class1_pdf, class2_pdf)

        # Retain feature if KL divergence is above the threshold
        if kl_div > threshold:
            retained_features.append(feature)
            
    return retained_features

In [6]:
def select_k_best_features(df, features, class_col, k=10):
    """
    Selects the top k features using SelectKBest based on the ANOVA F-test.

    Parameters:
        df (pd.DataFrame): The dataset containing the features and class column.
        features (list): List of feature column names.
        class_col (str): Name of the target class column.
        k (int): Number of top features to select.

    Returns:
        list: List of the retained features.
    """
    # Separate features and target
    X = df[features]
    y = df[class_col]

    # Apply SelectKBest with ANOVA F-test
    selector = SelectKBest(score_func=f_classif, k=k)
    selector.fit(X, y)

    # Get the indices of the selected features
    selected_indices = selector.get_support(indices=True)

    # Get the names of the selected features
    retained_features = [features[i] for i in selected_indices]

    return retained_features

In [7]:
def plot_confusion_matrix(y_test, y_pred, class_names=None):
    # Generate the confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    # Normalize the confusion matrix by row (true labels)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Plot the confusion matrix
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    
    plt.title('Confusion Matrix (Normalized)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

In [ ]:
# Replace NaNs with the mean value of each column
df.fillna(df.mean(), inplace=True)

# features = df.columns[:len(df.columns)-2].tolist()
features = df.columns[:len(df.columns)-3].tolist()

print("last feature in the list:", features[-1])
print(len(features))

features = reduce_features(df, features, variance_threshold=0.1, max_corr=0.8)
print(len(features))
# kept_features = filter_features_by_distribution(df, features, class_col='walk_mode', threshold=0.2)
# kept_features = select_k_best_features(df, features, 'walk_mode', k=10)
# print(len(kept_features))
# features = kept_features

In [ ]:
y = df["walk_mode"]
# Use LabelEncoder to convert string classes to numeric labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

participants = df["participant_id"].unique()

random.seed(2)
random.shuffle(participants)

# Assign a certain percentage of participants to the training set and the remaining to the test set
train_percentage = 0.8
num_train = int(train_percentage * len(participants))

train_participants = participants[:num_train]
test_participants = participants[num_train:]

# Use the participant list to filter the dataset into training and testing sets
train_set = df[df["participant_id"].isin(train_participants)]
test_set = df[df["participant_id"].isin(test_participants)]

# Split the data into features (X) and labels (y)
X_train, y_train = train_set[features], train_set["walk_mode"]
X_test, y_test = test_set[features], test_set["walk_mode"]

# X_train, X_test, y_train, y_test = train_test_split(df[features], y, test_size=0.2, stratify=y, random_state=42)

# Scale the data (not necessary with xgb)
# sc = MinMaxScaler()
# X_train = sc.fit_transform(X_train)
# X_test = sc.transform(X_test)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

####### only if classes start from 1 instead of 0 #######
# y_train = y_train - 1
# y_test = y_test - 1

# Hyperparameter tuning using GridSearchCV
# param_grid = {
#     "learning_rate": [0.1, 0.01, 0.001],
#     "max_depth": [3, 5, 10],
#     "n_estimators": [10, 40, 100, 150],
# }

param_grid = {
    "learning_rate": [0.1],
    "max_depth": [3, 5],
    "n_estimators": [100, 150],
}

# Initialize StratifiedGroupKFold
stratified_group_kfold = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2)

# Perform grid search with stratified subject-wise cross-validation
grid_search = GridSearchCV(
    estimator=XGBClassifier(),
    param_grid=param_grid,
    cv=stratified_group_kfold.split(X_train, y_train, groups=train_set["participant_id"]),
    scoring="accuracy",
    n_jobs=-1,
)

# Fit the grid search model
grid_search.fit(X_train, y_train)

# Best parameters
best_params = grid_search.best_params_
print("Best hyperparameters:", best_params)
# Compute cross-validation scores
cv_scores = cross_val_score(
    XGBClassifier(**best_params), 
    # XGBClassifier(), 
    X_train, 
    y_train, 
    cv=stratified_group_kfold.split(X_train, y_train, groups=train_set["participant_id"]), 
    scoring="accuracy", 
    n_jobs=-1
)

# Print results
print("Cross-validation scores:", cv_scores)
print("Mean cross-validation score:", cv_scores.mean())

# Train the final model with the best hyperparameters
final_model = XGBClassifier(**best_params)
# final_model = XGBClassifier()

final_model.fit(X_train, y_train)

# Evaluate the model on the test set
y_pred = final_model.predict(X_test)
classification_rep = classification_report(y_test, y_pred)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

# F1 Score (can use 'macro', 'micro', or 'weighted' average depending on preference)
f1 = f1_score(y_test, y_pred, average='macro')  # 'macro' averages F1 across all classes equally

# Sensitivity (Recall) - using 'macro' to calculate recall for each class and average
sensitivity = recall_score(y_test, y_pred, average='macro')  # Recall averaged across all classes

# Specificity - need to calculate per class and average
conf_matrix = confusion_matrix(y_test, y_pred)
specificity_per_class = []

for i in range(conf_matrix.shape[0]):
    # True Negatives are all the elements that are not in the i-th row and i-th column
    tn = np.sum(conf_matrix) - (np.sum(conf_matrix[i, :]) + np.sum(conf_matrix[:, i]) - conf_matrix[i, i])
    fp = np.sum(conf_matrix[:, i]) - conf_matrix[i, i]
    fn = np.sum(conf_matrix[i, :]) - conf_matrix[i, i]
    tp = conf_matrix[i, i]
    
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0  # Avoid division by zero
    specificity_per_class.append(specificity)

# Average specificity over all classes
specificity = np.mean(specificity_per_class)

# Print results
print(f"Accuracy: {accuracy:.2f}")
print(f"F1 Score (Macro): {f1:.2f}")
print(f"Sensitivity (Recall - Macro): {sensitivity:.2f}")
print(f"Specificity (Average): {specificity:.2f}")
print("Classification Report:\n", classification_rep)

# Get feature importance scores
feature_importance = final_model.feature_importances_

# Create a DataFrame with feature names and their importance scores
feature_importance_df = pd.DataFrame(
    {"Feature": features, "Importance": feature_importance}
)

# Sort the DataFrame by importance in descending order
feature_importance_df = feature_importance_df.sort_values(
    by="Importance", ascending=False
)

# Select the top 5 features
top_5_features = feature_importance_df.head(5)

# Plot the top 5 features
plt.figure(figsize=(12, 6))
plt.barh(top_5_features["Feature"][::-1], top_5_features["Importance"][::-1])
plt.xlabel("Feature Importance")
plt.title("Top 5 Features Of The XGBoost Model")
plt.show()

plot_confusion_matrix(y_test, y_pred, unique_vals)

In [ ]:
y = df["walk_mode"]
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)
participants = df["participant_id"].unique()
accuracies = []
for i in range(1, 10):
    random.seed(i)
    random.shuffle(participants)
    train_percentage = 0.8
    num_train = int(train_percentage * len(participants))
    train_participants = participants[:num_train]
    test_participants = participants[num_train:]
    train_set = df[df["participant_id"].isin(train_participants)]
    test_set = df[df["participant_id"].isin(test_participants)]
    X_train, y_train = train_set[features], train_set["walk_mode"]
    X_test, y_test = test_set[features], test_set["walk_mode"]
    param_grid = {
        "learning_rate": [0.1],
        "max_depth": [3, 5],
        "n_estimators": [100, 150],
    }
    stratified_group_kfold = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2)
    grid_search = GridSearchCV(
        estimator=XGBClassifier(),
        param_grid=param_grid,
        cv=stratified_group_kfold.split(X_train, y_train, groups=train_set["participant_id"]),
        scoring="accuracy",
        n_jobs=-1,
    )
    grid_search.fit(X_train, y_train)
    best_params = grid_search.best_params_
    final_model = XGBClassifier(**best_params)
    final_model.fit(X_train, y_train)
    y_pred = final_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy:.2f}")
    accuracies.append(accuracy)
print('Mean accuracy:', np.mean(accuracies))
print('Accuracy standard deviation:', np.std(accuracies))